# E1.7 · Continuous control verification

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.6 · Operating vs outcome guardrails](https://spbreed.github.io/cyber-commons/lessons/E1.6.html)**.

| | |
|---|---|
| Tools used | OPA, OSCAL, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Automate one evidence package on a schedule.

**Why a security engineer needs it.** Automating judgment instead of evidence collection. The control it builds is: agent-assisted evidence collection, drift detection, exception tracking.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A control that is verified annually is a control you know about once a year. Continuous verification is the only version of assurance that keeps up with a system whose behaviour changes between tests.

> **At CyberTravels.** A control verified once a year on a system whose prompt changed on Tuesday. Continuous verification is the only version of assurance that keeps up with CyberTravels.

## 2 · The framework

```
   annual                          continuous
   +-------------+                 +-------------------------+
   | one sample  |                 | probe on every change   |
   | one date    |      vs         | sample continuously     |
   | one signature|                | escalate on failure     |
   +-------------+                 +-------------------------+

   assurance that keeps up with a system that changes weekly
```

Continuous control verification is the operating model that follows from E1.1.

The number that matters is not how much passed once. It is **how much is
currently evidenced** — controls whose most recent test is passing *and* within
its freshness window.

Three states, and the third is the one classical GRC tooling cannot express:

- **PASS** — tested, passing, in window.
- **FAIL** — tested, failing. Honest and actionable.
- **STALE** — tested, was passing, out of window. **Not a pass.**

Plus the absence state: no evidence at all, which is different from failing and
is often the largest category in a first assessment.

## 3 · Collecting the runtime evidence, as a skill

Automating a control means something has to go and look. For the network and logging controls that underwrite every default-deny claim CyberTravels makes, that is a posture collector: egress rules, private endpoints, route tables, key policies, and whether the audit trail is not merely enabled but **delivering**. It collects; it does not conclude. This is the file in this repository:

### The skill — [`skills/attestation/aws-runtime-posture-collector/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/attestation/aws-runtime-posture-collector/SKILL.md)

```yaml
name: aws-runtime-posture-collector
description: >-
  Snapshot a deployment's cloud network, crypto and logging posture as
  evidence for default-deny and egress controls. Use to evidence network
  isolation, to check whether private endpoints and endpoint policies are in
  place, or to record the runtime configuration an attestation depends on.
allowed-tools: Bash, Read
```

# Aws Runtime Posture Collector

**Controls:** Controls 1 and 2 — runtime posture

## What this collects

Configuration state, at a point in time, for the network and crypto boundary
around one deployment. It is evidence for other skills' verdicts rather than a
verdict in itself.

## Procedure

1. **Security groups and network ACLs.** Enumerate egress rules. Any rule
   permitting `0.0.0.0/0` outbound is an egress-open finding regardless of what
   an application-layer policy says.
2. **Private endpoints.** Record whether a private endpoint exists for each
   model and gateway service in use, and read the endpoint policy — an
   unscoped endpoint policy is an endpoint that permits any principal.
3. **Route tables.** Identify NAT and internet gateways on the deployment's
   subnets. A private endpoint does not help if a default route to an internet
   gateway remains.
4. **Key policies.** Record key policies and condition keys that scope use to a
   specific service. Unconditioned key access is a finding.
5. **Logging.** Confirm the audit trail and log destinations are enabled and
   delivering, and record the retention.

## Output contract

```json
{
  "deployment_id": "str",
  "collected_at": "str",
  "network": {
    "egress_open_findings": [{"sg": "str", "rule": "str"}],
    "private_endpoints": [{"service": "str", "present": true, "policy_scoped": true}],
    "internet_route_present": false
  },
  "crypto": {"keys": [{"id": "str", "conditioned": true}]},
  "logging": {"audit_trail_enabled": true, "log_destinations": ["str"], "retention_days": 0},
  "verdict": "PASS|PARTIAL|FAIL"
}
```

## Failure modes

- **Reading configuration and calling it enforcement.** This skill records what
  is configured. Whether traffic actually obeys it is the egress verifier's job.
- **Ignoring the route table** because a private endpoint exists.
- **Recording that logging is enabled** without checking that it is delivering.

In [ ]:
# The code is not in this notebook. It is the file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/attestation/aws-runtime-posture-collector/scripts/aws_runtime_posture_collector.py
SCRIPT = "skills/attestation/aws-runtime-posture-collector/scripts/aws_runtime_posture_collector.py"

import glob, os, subprocess, sys

# The skills tree: the attached dataset on Kaggle, the checkout locally.
_ROOTS = sorted(glob.glob("/kaggle/input/**/cyber-commons-skills", recursive=True)) + [".", "..", "../.."]
_root = next((r for r in _ROOTS if os.path.isfile(os.path.join(r, SCRIPT))), None)
if _root is None:
    raise SystemExit("skills tree not found. On Kaggle add the dataset "
                     "cybercommons/cyber-commons-skills; locally run from a checkout.")

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The skill loads and reports its shape, and the line to take from it is the boundary it draws: configuration is not enforcement. A private endpoint next to a route table with a NAT gateway is a recorded fact and an open path at the same time, and logging that is switched on but not delivering evidences nothing at all.

## Your turn

Automate the control with the shortest freshness window first — it is the one costing the most manual effort and going stale most often. One automated test converts an annual assertion into a live control.

---

**Next → [E1.8 · Third-party and model supply chain risk](https://spbreed.github.io/cyber-commons/lessons/E1.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*